# Notebook 1 — Data Cleaning & Exploratory Data Analysis

**What this notebook does:** Loads the raw Telco customer churn dataset, cleans it
(fixes `TotalCharges` type, drops bad rows, removes duplicates, recodes columns),
prints a data quality report, then produces 9 EDA charts exploring what drives churn.

**Input:** `data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv`

**Output:**
- `data/processed/cleaned_churn.csv`
- Charts saved to `outputs/charts/eda/`

**Estimated run time:** ~30 seconds

In [1]:
# Imports and shared project utilities
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..")))
from utils import set_style, save_chart, save_dataframe, load_csv_safely, annotate_vertical_bars, annotate_horizontal_bars, project_path

set_style()

## 1. Load the raw data

In [2]:
# Load the raw CSV and inspect its basic shape and structure
raw_path = project_path("data", "raw", "WA_Fn-UseC_-Telco-Customer-Churn.csv")
try:
    df = pd.read_csv(raw_path)
except FileNotFoundError:
    print(f"❌ File not found: {raw_path}")
    print("   Please download the dataset from Kaggle (blastchar/telco-customer-churn) "
          "and place it at data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv")
    raise

print("Shape:", df.shape)
print("\nData types:\n", df.dtypes)
print("\nFirst 5 rows:")
df.head()

Shape: (7043, 21)

Data types:
 customerID           object
gender               object
SeniorCitizen         int64
Partner              object
Dependents           object
tenure                int64
PhoneService         object
MultipleLines        object
InternetService      object
OnlineSecurity       object
OnlineBackup         object
DeviceProtection     object
TechSupport          object
StreamingTV          object
StreamingMovies      object
Contract             object
PaperlessBilling     object
PaymentMethod        object
MonthlyCharges      float64
TotalCharges         object
Churn                object
dtype: object

First 5 rows:


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [3]:
# Null counts before cleaning
print("Null counts (raw):")
print(df.isnull().sum())

Null counts (raw):
customerID          0
gender              0
SeniorCitizen       0
Partner             0
Dependents          0
tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
Contract            0
PaperlessBilling    0
PaymentMethod       0
MonthlyCharges      0
TotalCharges        0
Churn               0
dtype: int64


## 2. Clean the data

In [4]:
# TotalCharges is loaded as a string column because a handful of rows contain
# blank strings (customers with 0 tenure). Coerce to numeric so those become NaN.
original_shape = df.shape
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

rows_with_nan_totalcharges = df["TotalCharges"].isnull().sum()
print(f"Rows with non-numeric TotalCharges (now NaN): {rows_with_nan_totalcharges}")

df = df.dropna(subset=["TotalCharges"])
print(f"Dropped {rows_with_nan_totalcharges} rows with missing TotalCharges")
print("Shape after dropping NaN rows:", df.shape)

Rows with non-numeric TotalCharges (now NaN): 11
Dropped 11 rows with missing TotalCharges
Shape after dropping NaN rows: (7032, 21)


In [5]:
# Encode the target: Yes -> 1, No -> 0
df["Churn"] = df["Churn"].map({"Yes": 1, "No": 0})

# Recode SeniorCitizen from 0/1 to No/Yes for readability, consistent with the
# other categorical Yes/No columns in the dataset
df["SeniorCitizen"] = df["SeniorCitizen"].map({0: "No", 1: "Yes"})

In [6]:
# Remove duplicate rows, if any
rows_before = df.shape[0]
df = df.drop_duplicates()
duplicates_removed = rows_before - df.shape[0]
print(f"Duplicate rows removed: {duplicates_removed}")

Duplicate rows removed: 0


## 3. Data quality report

In [7]:
# Summarise the effect of cleaning and confirm the data is now analysis-ready
cleaned_shape = df.shape
churned_count = df["Churn"].sum()
total_count = df.shape[0]
churn_rate = churned_count / total_count * 100

print("=" * 60)
print("DATA QUALITY REPORT")
print("=" * 60)
print(f"Original shape : {original_shape}")
print(f"Cleaned shape  : {cleaned_shape}")
print(f"\nColumn data types:\n{df.dtypes}")
print(f"\nChurn rate: {churn_rate:.1f}% of customers churned ({churned_count} out of {total_count})")
print(f"\nNull counts after cleaning:\n{df.isnull().sum()}")
print(f"\nTotal nulls remaining: {df.isnull().sum().sum()} (should be 0)")
print("=" * 60)

DATA QUALITY REPORT
Original shape : (7043, 21)
Cleaned shape  : (7032, 21)

Column data types:
customerID           object
gender               object
SeniorCitizen        object
Partner              object
Dependents           object
tenure                int64
PhoneService         object
MultipleLines        object
InternetService      object
OnlineSecurity       object
OnlineBackup         object
DeviceProtection     object
TechSupport          object
StreamingTV          object
StreamingMovies      object
Contract             object
PaperlessBilling     object
PaymentMethod        object
MonthlyCharges      float64
TotalCharges        float64
Churn                 int64
dtype: object

Churn rate: 26.6% of customers churned (1869 out of 7032)

Null counts after cleaning:
customerID          0
gender              0
SeniorCitizen       0
Partner             0
Dependents          0
tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity  

In [8]:
# Save the cleaned dataset for use in downstream notebooks
cleaned_path = project_path("data", "processed", "cleaned_churn.csv")
save_dataframe(df, cleaned_path)

✅ Saved: /Users/prasadlola/Documents/customer_churn_project/data/processed/cleaned_churn.csv (946.7KB) — 7032 rows, 21 columns


## 4. Exploratory Data Analysis

### Chart 1 — Overall churn distribution

In [9]:
# Side-by-side pie chart and count bar chart of the churn target
fig, axes = plt.subplots(1, 2, figsize=(12, 6))

churn_labels = df["Churn"].map({0: "No", 1: "Yes"})
counts = churn_labels.value_counts()

axes[0].pie(counts, labels=counts.index, autopct="%1.1f%%", colors=["#4C72B0", "#DD8452"],
            startangle=90)
axes[0].set_title("Churn Share")

sns.countplot(x=churn_labels, order=["No", "Yes"], hue=churn_labels, palette=["#4C72B0", "#DD8452"],
              legend=False, ax=axes[1])
axes[1].set_title("Churn Counts")
axes[1].set_xlabel("Churn")
axes[1].set_ylabel("Number of Customers")
annotate_vertical_bars(axes[1], fmt="{:.0f}")

fig.suptitle("Overall Churn Distribution", fontsize=14, fontweight="bold")
save_chart(fig, project_path("outputs", "charts", "eda", "chart1_churn_distribution.png"))

✅ Saved: /Users/prasadlola/Documents/customer_churn_project/outputs/charts/eda/chart1_churn_distribution.png (68.9KB)


### Chart 2 — Churn rate by Contract type

In [10]:
# Bar chart of churn rate (%) for each contract type, annotated with exact values
churn_by_contract = df.groupby("Contract")["Churn"].mean().mul(100).round(1).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(12, 6))
sns.barplot(x=churn_by_contract.index, y=churn_by_contract.values, hue=churn_by_contract.index,
            palette="Blues_d", legend=False, ax=ax)
ax.set_title("Churn Rate by Contract Type", fontsize=14, fontweight="bold")
ax.set_xlabel("Contract Type")
ax.set_ylabel("Churn Rate (%)")
annotate_vertical_bars(ax, fmt="{:.1f}%")

save_chart(fig, project_path("outputs", "charts", "eda", "chart2_churn_by_contract.png"))

✅ Saved: /Users/prasadlola/Documents/customer_churn_project/outputs/charts/eda/chart2_churn_by_contract.png (48.4KB)


### Chart 3 — Churn rate by Internet Service

In [11]:
# Bar chart of churn rate (%) for each internet service type
churn_by_internet = df.groupby("InternetService")["Churn"].mean().mul(100).round(1).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(12, 6))
sns.barplot(x=churn_by_internet.index, y=churn_by_internet.values, hue=churn_by_internet.index,
            palette="Oranges_d", legend=False, ax=ax)
ax.set_title("Churn Rate by Internet Service Type", fontsize=14, fontweight="bold")
ax.set_xlabel("Internet Service")
ax.set_ylabel("Churn Rate (%)")
annotate_vertical_bars(ax, fmt="{:.1f}%")

save_chart(fig, project_path("outputs", "charts", "eda", "chart3_churn_by_internet_service.png"))

✅ Saved: /Users/prasadlola/Documents/customer_churn_project/outputs/charts/eda/chart3_churn_by_internet_service.png (47.4KB)


### Chart 4 — Churn rate by Payment Method

In [12]:
# Horizontal bar chart since payment method labels are long
churn_by_payment = df.groupby("PaymentMethod")["Churn"].mean().mul(100).round(1).sort_values()

fig, ax = plt.subplots(figsize=(12, 6))
sns.barplot(x=churn_by_payment.values, y=churn_by_payment.index, hue=churn_by_payment.index,
            palette="Greens_d", legend=False, ax=ax)
ax.set_title("Churn Rate by Payment Method", fontsize=14, fontweight="bold")
ax.set_xlabel("Churn Rate (%)")
ax.set_ylabel("Payment Method")
annotate_horizontal_bars(ax, fmt="{:.1f}%")

save_chart(fig, project_path("outputs", "charts", "eda", "chart4_churn_by_payment_method.png"))

✅ Saved: /Users/prasadlola/Documents/customer_churn_project/outputs/charts/eda/chart4_churn_by_payment_method.png (56.0KB)


### Chart 5 — Tenure distribution: churned vs retained

In [13]:
# Overlapping histogram comparing tenure for churned vs retained customers
fig, ax = plt.subplots(figsize=(12, 6))
ax.hist(df.loc[df["Churn"] == 0, "tenure"], bins=30, alpha=0.6, label="Retained", color="#4C72B0")
ax.hist(df.loc[df["Churn"] == 1, "tenure"], bins=30, alpha=0.6, label="Churned", color="#DD8452")
ax.set_title("Tenure Distribution: Churned vs Retained", fontsize=14, fontweight="bold")
ax.set_xlabel("Tenure (months)")
ax.set_ylabel("Number of Customers")
ax.legend()

save_chart(fig, project_path("outputs", "charts", "eda", "chart5_tenure_distribution.png"))

✅ Saved: /Users/prasadlola/Documents/customer_churn_project/outputs/charts/eda/chart5_tenure_distribution.png (55.3KB)


### Chart 6 — Monthly Charges: churned vs retained

In [14]:
# Side-by-side boxplot of MonthlyCharges split by churn status
fig, ax = plt.subplots(figsize=(12, 6))
sns.boxplot(x=df["Churn"].map({0: "Retained", 1: "Churned"}), y=df["MonthlyCharges"],
            hue=df["Churn"].map({0: "Retained", 1: "Churned"}), palette=["#4C72B0", "#DD8452"],
            legend=False, ax=ax)
ax.set_title("Monthly Charges: Churned vs Retained", fontsize=14, fontweight="bold")
ax.set_xlabel("Churn Status")
ax.set_ylabel("Monthly Charges (£)")

save_chart(fig, project_path("outputs", "charts", "eda", "chart6_monthly_charges_boxplot.png"))

✅ Saved: /Users/prasadlola/Documents/customer_churn_project/outputs/charts/eda/chart6_monthly_charges_boxplot.png (45.5KB)


### Chart 7 — Churn rate by Senior Citizen status

In [15]:
# Bar chart comparing churn rate for seniors vs non-seniors
churn_by_senior = df.groupby("SeniorCitizen")["Churn"].mean().mul(100).round(1)

fig, ax = plt.subplots(figsize=(12, 6))
sns.barplot(x=churn_by_senior.index, y=churn_by_senior.values, hue=churn_by_senior.index,
            palette="Purples_d", legend=False, ax=ax)
ax.set_title("Churn Rate by Senior Citizen Status", fontsize=14, fontweight="bold")
ax.set_xlabel("Senior Citizen")
ax.set_ylabel("Churn Rate (%)")
annotate_vertical_bars(ax, fmt="{:.1f}%")

save_chart(fig, project_path("outputs", "charts", "eda", "chart7_churn_by_senior_citizen.png"))

✅ Saved: /Users/prasadlola/Documents/customer_churn_project/outputs/charts/eda/chart7_churn_by_senior_citizen.png (44.5KB)


### Chart 8 — Correlation heatmap

In [16]:
# Correlation heatmap across numeric columns only
numeric_df = df.select_dtypes(include=[np.number])

fig, ax = plt.subplots(figsize=(12, 8))
sns.heatmap(numeric_df.corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=ax)
ax.set_title("Correlation Heatmap of Numeric Features", fontsize=14, fontweight="bold")

save_chart(fig, project_path("outputs", "charts", "eda", "chart8_correlation_heatmap.png"))

✅ Saved: /Users/prasadlola/Documents/customer_churn_project/outputs/charts/eda/chart8_correlation_heatmap.png (74.3KB)


### Chart 9 — Number of services vs churn rate

In [17]:
# Build num_services: count of subscribed add-on/core services (Yes = 1, else 0)
service_columns = ["PhoneService", "MultipleLines", "OnlineSecurity", "OnlineBackup",
                   "DeviceProtection", "TechSupport", "StreamingTV", "StreamingMovies"]

df["num_services"] = df[service_columns].apply(lambda col: (col == "Yes").astype(int)).sum(axis=1)

churn_by_num_services = df.groupby("num_services")["Churn"].mean().mul(100).round(1)

fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(churn_by_num_services.index, churn_by_num_services.values, marker="o", linewidth=2, color="#4C72B0")
ax.set_title("Churn Rate by Number of Services Subscribed", fontsize=14, fontweight="bold")
ax.set_xlabel("Number of Services Subscribed")
ax.set_ylabel("Churn Rate (%)")
ax.set_xticks(range(0, 9))

save_chart(fig, project_path("outputs", "charts", "eda", "chart9_churn_by_num_services.png"))

✅ Saved: /Users/prasadlola/Documents/customer_churn_project/outputs/charts/eda/chart9_churn_by_num_services.png (85.6KB)


In [18]:
# Print supporting numbers referenced in the business insights below
print("Churn rate by contract type:\n", churn_by_contract, "\n")
print("Churn rate by internet service:\n", churn_by_internet, "\n")
print("Churn rate by payment method:\n", churn_by_payment, "\n")
print("Churn rate by senior citizen:\n", churn_by_senior, "\n")
print("Churn rate by number of services:\n", churn_by_num_services, "\n")
print("Median tenure - churned:", df.loc[df['Churn']==1, 'tenure'].median(),
      " retained:", df.loc[df['Churn']==0, 'tenure'].median())
print("Median MonthlyCharges - churned:", df.loc[df['Churn']==1, 'MonthlyCharges'].median(),
      " retained:", df.loc[df['Churn']==0, 'MonthlyCharges'].median())

Churn rate by contract type:
 Contract
Month-to-month    42.7
One year          11.3
Two year           2.8
Name: Churn, dtype: float64 

Churn rate by internet service:
 InternetService
Fiber optic    41.9
DSL            19.0
No              7.4
Name: Churn, dtype: float64 

Churn rate by payment method:
 PaymentMethod
Credit card (automatic)      15.3
Bank transfer (automatic)    16.7
Mailed check                 19.2
Electronic check             45.3
Name: Churn, dtype: float64 

Churn rate by senior citizen:
 SeniorCitizen
No     23.7
Yes    41.7
Name: Churn, dtype: float64 

Churn rate by number of services:
 num_services
0    43.8
1    21.2
2    32.9
3    36.5
4    31.4
5    25.6
6    22.6
7    12.4
8     5.3
Name: Churn, dtype: float64 

Median tenure - churned: 10.0  retained: 38.0
Median MonthlyCharges - churned: 79.65  retained: 64.45


## Key Business Insights from EDA

- **Finding:** Month-to-month customers churn at 42.7%, versus 11.3% for one-year and just 2.8% for two-year contracts. → **Business implication:** Contract length is the single strongest lever for retention — incentivising month-to-month customers to move to annual contracts (e.g. a modest discount for upgrading) should directly cut churn.

- **Finding:** Fiber optic customers churn at 41.9%, more than double the DSL rate (19.0%) and nearly six times the no-internet rate (7.4%), despite fiber being the premium/higher-margin product. → **Business implication:** Investigate fiber service quality and pricing complaints specifically — losing customers on the highest-revenue product is the costliest form of churn.

- **Finding:** Electronic check payers churn at 45.3%, far above credit card (15.3%), bank transfer (16.7%) and mailed check (19.2%). → **Business implication:** Migrating electronic-check customers to automatic payment methods (autopay incentives) is a low-cost intervention likely to reduce churn.

- **Finding:** Churned customers have a median tenure of just 10 months vs 38 months for retained customers, and median monthly charges of £79.65 vs £64.45 for retained customers. → **Business implication:** New, higher-paying customers are the highest-risk group — an onboarding/early-life retention program in the first 12 months would target the point of greatest revenue leakage.

- **Finding:** Senior citizens churn at 41.7%, nearly double the non-senior rate of 23.7%. → **Business implication:** Senior customers may need simplified plans, more accessible support, or targeted retention outreach.

- **Finding:** Customers with 6-8 subscribed services churn far less (22.6% → 5.3%) than customers with 0 services (43.8%), showing that service bundling is strongly associated with retention (with a secondary bump in churn around 2-3 services). → **Business implication:** Cross-selling add-on services (especially OnlineSecurity and TechSupport, which anchor the bundle) is a promising upsell-driven retention strategy, though the 2-3 service dip suggests bundling needs to reach a critical mass to lock in loyalty.
